# 🏥 Gemma 4 E2B: 災害時START式トリアージ LoRA Fine-Tuning & ブラウザ用エクスポート

Gemma Meetup 2026 登壇用パイプラインノートブックです。

このノートブックでは以下を一気に実行します：
1. **環境構築** (`peft`, `trl`, `transformers`, `optimum`, `litert-torch`)
2. **START式トリアージデータセットの生成** (救急判定ルール & 構造化JSON)
3. **Gemma 4 E2B-it の LoRA ファインチューニング**
4. **LoRA アダプタをベース重みに完全マージ** (`model.merge_and_unload()`)
5. **Transformers.js 用 ONNX (q4f16) エクスポート**
6. **LiteRT-LM 用 .litertlm エクスポート**
7. **ブラウザデモ用モデルの zip 圧縮・ダウンロード**

## 1. 依存ライブラリのインストール

In [ ]:
!pip install -q -U torch transformers peft trl datasets accelerate optimum[onnxruntime] litert-torch

## 2. Hugging Face へのログイン (Gemma モデル利用のため)
※ Gemma 4 の利用規約に同意したアカウントの Token を入力してください。

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. START式トリアージ学習データの作成

In [ ]:
import json
from pathlib import Path

DATASET = [
    {
        "input": "45歳男性。家屋倒壊現場から救出。自力歩行不可。呼吸数36回/分（頻呼吸）。橈骨動脈は微弱に触知。呼びかけに対してうめき声のみで従命不能。",
        "output": json.dumps({
            "triage_category": "RED",
            "priority": "最優先治療群",
            "vital_assessment": {"walkable": False, "respiratory_rate": 36, "radial_pulse": "weak", "obeys_commands": False},
            "judgment_reason": "自力歩行不可、呼吸数が30回/分以上、従命不能のため最優先治療対象。",
            "immediate_action": "気道確保、高濃度酸素投与、最優先搬送手配"
        }, ensure_ascii=False)
    },
    {
        "input": "28歳女性。ガラス片による大腿部深部裂傷、多量出血。自力歩行不可。呼吸数24回/分。橈骨動脈触知不可。爪床圧迫テスト(CRT)3秒。意識清明。",
        "output": json.dumps({
            "triage_category": "RED",
            "priority": "最優先治療群",
            "vital_assessment": {"walkable": False, "respiratory_rate": 24, "radial_pulse": "absent", "obeys_commands": True},
            "judgment_reason": "循環不全（橈骨動脈触知不可、CRT延長）による出血性ショックの危険。",
            "immediate_action": "大腿部直接圧迫止血（または止血帯適用）、下肢挙上、急速輸液路確保"
        }, ensure_ascii=False)
    },
    {
        "input": "52歳男性。落下物により右下腿変形・激痛。自力歩行不可。呼吸数20回/分。橈骨動脈正常に触知。意識清明、「痛くて立てない」と明確に受け答え可能。",
        "output": json.dumps({
            "triage_category": "YELLOW",
            "priority": "待機的治療群",
            "vital_assessment": {"walkable": False, "respiratory_rate": 20, "radial_pulse": "present", "obeys_commands": True},
            "judgment_reason": "歩行不能だが、呼吸(10-29回)・循環(橈骨動脈触知可)・意識(従命可)のバイタルは安定。",
            "immediate_action": "患部副木固定、クーリング、バイタル定期再評価"
        }, ensure_ascii=False)
    },
    {
        "input": "24歳男性。「歩ける方はこちらへ移動してください」の呼びかけに対し、自力で安全ゾーンまで歩行して集合。前腕に擦過傷あり。",
        "output": json.dumps({
            "triage_category": "GREEN",
            "priority": "保留群（軽症）",
            "vital_assessment": {"walkable": True, "respiratory_rate": None, "radial_pulse": None, "obeys_commands": True},
            "judgment_reason": "自力歩行可能のため一次トリアージではGREENと判定。",
            "immediate_action": "軽症待機エリアへの誘導、創部洗浄・止血、自己申告に基づく経過観察"
        }, ensure_ascii=False)
    },
    {
        "input": "55歳男性。コンクリート下敷き。自力歩行不可。自発呼吸なし。気道確保を実施したが、呼吸の再開なし。頸動脈・橈骨動脈ともに触知せず。",
        "output": json.dumps({
            "triage_category": "BLACK",
            "priority": "不処置（死亡・救命不能）",
            "vital_assessment": {"walkable": False, "respiratory_rate": 0, "radial_pulse": "absent", "obeys_commands": False},
            "judgment_reason": "気道確保後も無呼吸、脈拍触知不可。START基準における死亡群。",
            "immediate_action": "トリアージタッグ黒装着、他の生存患者の救命を優先"
        }, ensure_ascii=False)
    }
]

SYSTEM_PROMPT = "あなたは災害医療のSTART式トリアージ判定スペシャリストAIです。入力された傷病者のバイタル・身体状況に基づき、厳密にSTART基準に従って判定を行い、所定のJSON形式のみで回答してください。"

with open("triage_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in DATASET:
        record = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item["input"]},
                {"role": "model", "content": item["output"]}
            ]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("triage_dataset.jsonl created successfully!")

## 4. Gemma 4 E2B-it の LoRA 学習 & マージ

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, TaskType
from trl import SFTTrainer

MODEL_ID = "google/gemma-4-E2B-it"

# 1. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 2. Dataset
dataset = load_dataset("json", data_files="triage_dataset.jsonl", split="train")
def format_fn(sample):
    return {"text": tokenizer.apply_chat_template(sample["messages"], tokenize=False)}
formatted_dataset = dataset.map(format_fn)

# 3. Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 4. LoRA Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 5. Training
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config
)

trainer.train()

# 6. Merge weights
print("Merging LoRA weights into standalone model...")
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained("./merged_gemma4_triage")
tokenizer.save_pretrained("./merged_gemma4_triage")
print("Merged model saved to ./merged_gemma4_triage")

## 5. Transformers.js 用 ONNX (q4f16) エクスポート

In [ ]:
!optimum-cli export onnx \
  --model ./merged_gemma4_triage \
  --task text-generation-with-past \
  --dtype q4f16 \
  ./dist_onnx

## 6. LiteRT-LM 用 .litertlm エクスポート

In [ ]:
!python -m litert_torch.generative.export_hf \
  ./merged_gemma4_triage \
  ./dist_litert \
  --task text_generation \
  --bundle_litert_lm true \
  --externalize_embedder true \
  --quantization dynamic_wi4_afp32

## 7. 生成されたモデルを zip に固めてダウンロード
ダウンロードした zip を解凍して、`gemma-meetup/dist_onnx` および `gemma-meetup/dist_litert` に配置すれば、ブラウザデモ画面 (`04_custom_triage_demo/index.html`) からローカルで直接ロードされます。

In [ ]:
!zip -r custom_gemma4_triage_models.zip ./dist_onnx ./dist_litert

from google.colab import files
files.download("custom_gemma4_triage_models.zip")